# SQ-MIL: Ovarian Cancer WSI Classification via SMMILe

End-to-end pipeline: feature extraction → superpixels → 5-fold CV training (Stage 1 + Stage 2) → evaluation → heatmaps.

**Required Colab environment:** A100 40 GB (Pro+)
- Runtime → Change runtime type → GPU → A100
- Colab Pro+ gives 24-hr sessions (essential for full training)

**Estimated total runtime on A100:**
| Step | Time |
|---|---|
| Feature extraction (513 WSIs, Conch) | ~45 min |
| Superpixel generation | ~10 min |
| Stage 1 training (5 folds × 40 epochs) | ~8–10 hr |
| Stage 2 training + eval (5 folds × 20 epochs) | ~4–5 hr |
| Heatmap generation | ~15 min |
| **Total** | **~14–16 hr** |

## 0. Environment Check

In [ ]:
import subprocess, sys

# Verify A100 GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip())

gpu_name = result.stdout.strip()
if 'A100' not in gpu_name:
    print('\n⚠️  WARNING: Not running on A100.')
    print('   Go to: Runtime → Change runtime type → GPU → A100 (requires Colab Pro+)')
    print('   Training will work on other GPUs but will be significantly slower.')
else:
    print('\n✓ A100 confirmed — optimal for this workload.')

## 1. Mount Google Drive & Clone Repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('Drive mounted at /content/drive')

In [ ]:
import os

REPO_DIR = '/content/SQ-MIL'
REPO_URL = 'https://github.com/z-pan/SQ-MIL.git'
BRANCH   = 'claude/init-sq-mil-project-Nu1aU'

if os.path.isdir(REPO_DIR):
    print('Repo already cloned — pulling latest...')
    !git -C {REPO_DIR} fetch origin {BRANCH} && git -C {REPO_DIR} reset --hard origin/{BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
print('Working directory:', os.getcwd())

## 2. Install System & Python Dependencies

In [ ]:
# OpenSlide system library (required for pyramidal TIFF reading)
!apt-get install -qq openslide-tools libopenjp2-7
print('System packages installed.')

In [ ]:
!pip install -q \
    openslide-python \
    scikit-image \
    scikit-learn \
    tifffile \
    h5py \
    pyyaml \
    timm \
    transformers \
    huggingface_hub \
    einops \
    opencv-python-headless \
    ruff

# Restart runtime after install to ensure openslide loads correctly
print('\nPython packages installed.')
print('⚠️  If this is your first install, click Runtime → Restart session, then re-run from Cell 0.')

## 3. Configuration

**Edit the variables below to match your setup before running any training cells.**

In [ ]:
# ============================================================
# USER CONFIGURATION — edit these paths and settings
# ============================================================

# --- Data source ---
# True  → using pre-extracted HuggingFace embeddings (skip Steps 1 & 2)
# False → extracting features from raw WSI files (run Steps 1 & 2)
USE_HF_EMBEDDINGS = True

# --- Paths ---
WSI_DIR       = '/content/drive/MyDrive/SQ-MIL/wsi'        # raw *.tif files (only needed if USE_HF_EMBEDDINGS=False)
LABELS_CSV    = '/content/drive/MyDrive/SQ-MIL/labels.csv' # columns: slide_id, label
DATA_ROOT     = '/content/drive/MyDrive/SQ-MIL'
RESULTS_ROOT  = '/content/drive/MyDrive/SQ-MIL/results'

# --- HuggingFace embedding paths (only used when USE_HF_EMBEDDINGS=True) ---
# Point these to the folders you downloaded from zeyugao/SMMILe_Datasets
HF_EMBED_DIR  = '/content/drive/MyDrive/SQ-MIL/ovarian_conch/conch'
HF_SP_DIR     = '/content/drive/MyDrive/SQ-MIL/ovarian_conch/sp_conch_n16_c50_512'

# --- Encoder ---
# 'conch'    → 512-dim (HuggingFace token required for feature extraction)
# 'resnet50' → 1024-dim (no token required)
ENCODER       = 'conch'

# --- HuggingFace token (needed for ENCODER='conch' AND USE_HF_EMBEDDINGS=False) ---
HF_TOKEN      = ''   # e.g. 'hf_XXXXXXXXXXXX'

# --- Training ---
N_FOLDS       = 5
SEED          = 42
GPU_ID        = 0

# --- Configs ---
S1_CONFIG     = 'configs/ovarian_conch_s1.yaml'
S2_CONFIG     = 'configs/ovarian_conch_s2.yaml'

# --- Derived output dirs (do not edit) ---
import os

if USE_HF_EMBEDDINGS:
    EMBED_DIR    = HF_EMBED_DIR   # loader reads flat layout directly — no copy needed
    SUPERPIX_DIR = HF_SP_DIR
else:
    EMBED_DIR    = os.path.join(DATA_ROOT, 'embeddings')
    SUPERPIX_DIR = os.path.join(DATA_ROOT, 'superpixels')

SPLITS_DIR   = os.path.join(DATA_ROOT, 'splits')
S1_OUT       = os.path.join(RESULTS_ROOT, 'stage1')
S2_OUT       = os.path.join(RESULTS_ROOT, 'stage2')
HEATMAP_OUT  = os.path.join(RESULTS_ROOT, 'heatmaps')

for d in [SPLITS_DIR, S1_OUT, S2_OUT, HEATMAP_OUT]:
    os.makedirs(d, exist_ok=True)

print('Configuration loaded.')
print(f'  USE_HF_EMBEDDINGS : {USE_HF_EMBEDDINGS}')
print(f'  EMBED_DIR         : {EMBED_DIR}')
print(f'  SUPERPIX_DIR      : {SUPERPIX_DIR}')
print(f'  Encoder           : {ENCODER}')
print(f'  Folds             : {N_FOLDS}')

## 4. Pre-flight Validation

In [ ]:
import glob, pandas as pd

errors = []

# Check WSI files (only required when extracting features from scratch)
if not USE_HF_EMBEDDINGS:
    wsi_files = sorted(glob.glob(os.path.join(WSI_DIR, '*.tif')))
    if len(wsi_files) == 0:
        errors.append(f'No *.tif files found in WSI_DIR: {WSI_DIR}')
    else:
        print(f'✓ Found {len(wsi_files)} WSI files')
else:
    wsi_files = []
    print(f'✓ Skipping WSI check (USE_HF_EMBEDDINGS=True)')
    # Check HF dirs exist
    import os as _os
    for d, name in [(HF_EMBED_DIR, 'HF_EMBED_DIR'), (HF_SP_DIR, 'HF_SP_DIR')]:
        npy_count = len(list(_os.scandir(d))) if _os.path.isdir(d) else 0
        if npy_count == 0:
            errors.append(f'{name} is empty or missing: {d}')
        else:
            print(f'✓ {name}: {npy_count} files found')

# Check labels.csv
if not os.path.isfile(LABELS_CSV):
    errors.append(f'LABELS_CSV not found: {LABELS_CSV}')
else:
    df = pd.read_csv(LABELS_CSV)
    required_cols = {'slide_id', 'label'}
    if not required_cols.issubset(df.columns):
        errors.append(f'labels.csv missing columns: {required_cols - set(df.columns)}')
    else:
        class_counts = df['label'].value_counts().to_dict()
        print(f'✓ labels.csv: {len(df)} slides, classes: {class_counts}')

# Check configs
for cfg in [S1_CONFIG, S2_CONFIG]:
    if not os.path.isfile(cfg):
        errors.append(f'Config not found: {cfg}')
    else:
        print(f'✓ Config exists: {cfg}')

# Check HF token (only needed for Conch feature extraction)
if ENCODER == 'conch' and not USE_HF_EMBEDDINGS:
    if not HF_TOKEN:
        errors.append('HF_TOKEN is empty — required for Conch feature extraction.')
    else:
        print('✓ HuggingFace token provided.')

# Check GPU
import torch
if not torch.cuda.is_available():
    errors.append('No CUDA GPU detected. Enable GPU runtime first.')
else:
    print(f'✓ CUDA: {torch.cuda.get_device_name(GPU_ID)}  '
          f'({torch.cuda.get_device_properties(GPU_ID).total_memory / 1e9:.1f} GB)')

if errors:
    print('\n❌ Pre-flight FAILED:')
    for e in errors:
        print(f'  • {e}')
    raise SystemExit('Fix the issues above before proceeding.')
else:
    print('\n✓ All pre-flight checks passed.')

## 5. HuggingFace Login (Conch encoder only)

In [ ]:
if USE_HF_EMBEDDINGS:
    print('✓ Skipped — HuggingFace login not needed when USE_HF_EMBEDDINGS=True.')
    print('  (The Conch model is only required for feature extraction, which is skipped.)')
elif ENCODER == 'conch':
    if not HF_TOKEN:
        raise ValueError(
            'HF_TOKEN is empty. Set it in the config cell before running this cell.\n'
            'Get your token at: https://huggingface.co/settings/tokens\n'
            'Request model access at: https://huggingface.co/MahmoodLab/conch'
        )
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✓ HuggingFace login successful.')
else:
    print('✓ Skipped — encoder is resnet50, no HuggingFace login needed.')

In [ ]:
import subprocess, sys

def run(cmd, **kwargs):
    """Run a shell command, stream output line-by-line, raise on failure."""
    print('$', ' '.join(str(c) for c in cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, **kwargs)
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {proc.returncode}')

print('✓ run() helper defined.')

## Step 1: Feature Extraction *(skip if USE_HF_EMBEDDINGS=True)*

Tessellate each WSI into 512×512 patches at 20× and extract embeddings.

- Output: `{EMBED_DIR}/{slide_id}/` — one `.npy` per patch + `coords.csv`
- **Skip this step** if you set `USE_HF_EMBEDDINGS = True` in the config

In [ ]:
if USE_HF_EMBEDDINGS:
    print('✓ Skipped — using pre-extracted HuggingFace embeddings.')
else:
    done = [d for d in os.listdir(EMBED_DIR)
            if os.path.isdir(os.path.join(EMBED_DIR, d)) and
            os.path.isfile(os.path.join(EMBED_DIR, d, 'coords.csv'))]
    print(f'Embeddings already extracted: {len(done)} / {len(wsi_files)} slides')

In [ ]:
if USE_HF_EMBEDDINGS:
    print('✓ Skipped — using pre-extracted HuggingFace embeddings.')
elif len(done) < len(wsi_files):
    run([
        sys.executable, 'scripts/01_extract_features.py',
        '--wsi_dir',      WSI_DIR,
        '--output_dir',   EMBED_DIR,
        '--encoder_name', ENCODER,
        '--patch_size',   '512',
        '--step_size',    '512',
        '--level',        '0',
        '--batch_size',   '64',
        '--device',       f'cuda:{GPU_ID}',
    ])
    print('\n✓ Feature extraction complete.')
else:
    print('✓ Skipped — already complete.')

## Step 2: Superpixel Generation *(skip if USE_HF_EMBEDDINGS=True)*

Run SLIC on the spatial embedding grid to generate superpatches.

- Output: `{SUPERPIX_DIR}/{slide_id}.npy`
- **Skip this step** if you set `USE_HF_EMBEDDINGS = True` in the config

In [ ]:
if USE_HF_EMBEDDINGS:
    print('✓ Skipped — superpixels already included in HuggingFace download.')
else:
    sp_done = [f for f in os.listdir(SUPERPIX_DIR) if f.endswith('.npy')]
    print(f'Superpixels already generated: {len(sp_done)} / {len(wsi_files)} slides')

In [ ]:
if USE_HF_EMBEDDINGS:
    print('✓ Skipped — superpixels already included in HuggingFace download.')
elif len(sp_done) < len(wsi_files):
    run([
        sys.executable, 'scripts/02_generate_superpixels.py',
        '--embeddings_dir', EMBED_DIR,
        '--output_dir',     SUPERPIX_DIR,
        '--n_segments',     '25',
        '--compactness',    '50',
    ])
    print('\n✓ Superpixel generation complete.')
else:
    print('✓ Skipped — already complete.')

## Step 3: Prepare Cross-validation Splits

Generate patient-level stratified 5-fold CV splits (80/20 train+val/test; 90/10 train/val within train+val).

- Output: `{SPLITS_DIR}/splits_{fold}.csv` for fold 0–4

In [ ]:
split_files = sorted(glob.glob(os.path.join(SPLITS_DIR, 'splits_*.csv')))
print(f'Split files found: {len(split_files)} / {N_FOLDS}')
for sf in split_files:
    df_s = pd.read_csv(sf)
    print(f'  {os.path.basename(sf)}: {dict(df_s["split"].value_counts())}')

In [ ]:
if len(split_files) < N_FOLDS:
    run([
        sys.executable, 'scripts/03_prepare_splits.py',
        '--labels',      LABELS_CSV,
        '--output_dir',  SPLITS_DIR,
        '--n_folds',     str(N_FOLDS),
        '--val_ratio',   '0.10',
        '--seed',        str(SEED),
    ])
    print('\n✓ Splits generated.')
else:
    print('✓ Skipped — all split files already exist.')

## Step 3b: Validate Downloaded Embeddings *(HuggingFace users only)*

**Run this section if `USE_HF_EMBEDDINGS = True`.**

Checks that every `slide_id` in `labels.csv` has a matching file in the embedding and superpixel directories, and saves a filtered `labels.csv` containing only the aligned slides.

The training code reads the flat HuggingFace layout directly — **no reorganization needed.**

In [ ]:
if not USE_HF_EMBEDDINGS:
    print('Skipped — only needed when USE_HF_EMBEDDINGS=True.')
else:
    # The HF_EMBED_DIR and HF_SP_DIR are already set as EMBED_DIR / SUPERPIX_DIR in the config
    print('Step 3b config:')
    print(f'  labels   : {LABELS_CSV}')
    print(f'  emb_dir  : {EMBED_DIR}')
    print(f'  sp_dir   : {SUPERPIX_DIR}')

In [ ]:
# Step 3b-i: Validate alignment (read-only)
if USE_HF_EMBEDDINGS:
    run([
        sys.executable, 'scripts/00_validate_data.py',
        '--labels',  LABELS_CSV,
        '--emb_dir', EMBED_DIR,
        '--sp_dir',  SUPERPIX_DIR,
        '--inspect',   # show internal structure of sample .npy files
    ])
else:
    print('Skipped.')

In [ ]:
# Step 3b-ii: Save a filtered labels.csv (only slides present in both emb + sp dirs)
# No --reorganize needed: the training loader reads flat layout directly.
if USE_HF_EMBEDDINGS:
    run([
        sys.executable, 'scripts/00_validate_data.py',
        '--labels',               LABELS_CSV,
        '--emb_dir',              EMBED_DIR,
        '--sp_dir',               SUPERPIX_DIR,
        '--save_filtered_labels', LABELS_CSV,  # overwrites in-place with aligned subset
    ])
    print(f'\n✓ labels.csv updated with aligned slides only: {LABELS_CSV}')
else:
    print('Skipped.')

## Step 4: Stage 1 Training (5 Folds)

Train the primary MIL network (NIC + gated attention + InD + InS) for 40 epochs per fold.

- Loss: BCE classification (L_cls)
- Checkpoints saved to `{S1_OUT}/fold{N}/best_model.pth`
- **Resume-safe**: folds with an existing `best_model.pth` are skipped

In [ ]:
for fold in range(N_FOLDS):
    ckpt = os.path.join(S1_OUT, f'fold{fold}', 'best_model.pth')
    status = '✓ done' if os.path.isfile(ckpt) else '○ pending'
    print(f'  Stage 1 fold {fold}: {status}')

In [ ]:
for fold in range(N_FOLDS):
    ckpt = os.path.join(S1_OUT, f'fold{fold}', 'best_model.pth')
    if os.path.isfile(ckpt):
        print(f'Stage 1 fold {fold}: already trained — skipping.')
        continue

    print(f'\n=== Stage 1 | Fold {fold} ===')
    run([
        sys.executable, 'scripts/train.py',
        '--config',     S1_CONFIG,
        '--stage',      '1',
        '--fold',       str(fold),
        '--seed',       str(SEED),
        '--data_root',  DATA_ROOT,
        '--output_dir', S1_OUT,
        '--gpu_id',     str(GPU_ID),
    ])
    print(f'✓ Stage 1 fold {fold} complete.')

print('\n✓ Stage 1 training complete for all folds.')

## Step 5: Stage 2 Training + Evaluation (5 Folds)

Refine with instance refinement layers (N=3) and MRF spatial constraint for 20 epochs per fold, then run evaluation.

- Loss: L_cls + L_ref + L_mrf
- Loads Stage 1 checkpoint for each fold
- Evaluation outputs: `{S2_OUT}/fold{N}/metrics.json` + `{S2_OUT}/fold{N}/*_inst.csv`
- **Resume-safe**: folds with an existing `best_model.pth` are skipped

In [ ]:
for fold in range(N_FOLDS):
    s1_ckpt = os.path.join(S1_OUT, f'fold{fold}', 'best_model.pth')
    s2_ckpt = os.path.join(S2_OUT, f'fold{fold}', 'best_model.pth')
    s1_status = '✓' if os.path.isfile(s1_ckpt) else '✗ MISSING'
    s2_status = '✓ done' if os.path.isfile(s2_ckpt) else '○ pending'
    print(f'  Fold {fold}: Stage1={s1_status}  Stage2={s2_status}')

In [ ]:
for fold in range(N_FOLDS):
    s1_ckpt = os.path.join(S1_OUT, f'fold{fold}', 'best_model.pth')
    s2_ckpt = os.path.join(S2_OUT, f'fold{fold}', 'best_model.pth')

    if not os.path.isfile(s1_ckpt):
        raise FileNotFoundError(
            f'Stage 1 checkpoint missing for fold {fold}: {s1_ckpt}\n'
            'Run Stage 1 training first (Cell above).'
        )

    if os.path.isfile(s2_ckpt):
        print(f'Stage 2 fold {fold}: already trained — skipping.')
    else:
        print(f'\n=== Stage 2 | Fold {fold} ===')
        run([
            sys.executable, 'scripts/train.py',
            '--config',      S2_CONFIG,
            '--stage',       '2',
            '--fold',        str(fold),
            '--seed',        str(SEED),
            '--stage1_ckpt', s1_ckpt,
            '--data_root',   DATA_ROOT,
            '--output_dir',  S2_OUT,
            '--gpu_id',      str(GPU_ID),
        ])
        print(f'✓ Stage 2 fold {fold} complete.')

    # Always run evaluation (idempotent)
    metrics_file = os.path.join(S2_OUT, f'fold{fold}', 'metrics.json')
    print(f'\n--- Evaluation | Fold {fold} ---')
    run([
        sys.executable, 'scripts/train.py',
        '--config',    S2_CONFIG,
        '--stage',     'eval',
        '--fold',      str(fold),
        '--ckpt',      s2_ckpt,
        '--data_root', DATA_ROOT,
        '--output_dir', S2_OUT,
        '--gpu_id',    str(GPU_ID),
    ])
    print(f'✓ Evaluation fold {fold} complete.')

print('\n✓ Stage 2 training + evaluation complete for all folds.')

## Step 6: Aggregate Metrics

Collect per-fold metrics and display mean ± std across 5 folds.

In [ ]:
import json
import numpy as np
import pandas as pd

metric_keys = ['wsi_auc', 'spatial_auc', 'spatial_f1', 'spatial_acc']
records = []

for fold in range(N_FOLDS):
    mfile = os.path.join(S2_OUT, f'fold{fold}', 'metrics.json')
    if not os.path.isfile(mfile):
        print(f'⚠ metrics.json missing for fold {fold}: {mfile}')
        continue
    with open(mfile) as f:
        m = json.load(f)
    row = {'fold': fold}
    row.update({k: m.get(k, float('nan')) for k in metric_keys})
    records.append(row)

if not records:
    print('No metrics found. Run evaluation first (Step 5).')
else:
    df_m = pd.DataFrame(records).set_index('fold')
    summary = pd.DataFrame({
        'mean': df_m.mean(),
        'std':  df_m.std(),
    }).T
    print('=== Per-fold metrics ===')
    print(df_m.to_string(float_format='{:.4f}'.format))
    print('\n=== Summary (mean ± std) ===')
    for col in metric_keys:
        mu  = df_m[col].mean()
        std = df_m[col].std()
        print(f'  {col:20s}: {mu*100:.2f} ± {std*100:.2f} %')

    # Save aggregate
    agg_path = os.path.join(RESULTS_ROOT, 'aggregate_metrics.csv')
    df_m.to_csv(agg_path)
    print(f'\nSaved to: {agg_path}')

In [ ]:
import matplotlib
matplotlib.use('Agg')   # non-interactive backend
import matplotlib.pyplot as plt

if records:
    fig, axes = plt.subplots(1, len(metric_keys), figsize=(16, 4))
    folds = df_m.index.tolist()

    for ax, key in zip(axes, metric_keys):
        values = df_m[key].tolist()
        mu  = np.mean(values)
        ax.bar(folds, [v * 100 for v in values], color='steelblue', alpha=0.8)
        ax.axhline(mu * 100, color='red', linestyle='--', label=f'mean={mu*100:.2f}%')
        ax.set_title(key.replace('_', ' ').title())
        ax.set_xlabel('Fold')
        ax.set_ylabel('%')
        ax.set_ylim(0, 105)
        ax.legend(fontsize=8)

    plt.suptitle('SQ-MIL: 5-fold Cross-validation Results', fontsize=13, y=1.02)
    plt.tight_layout()

    plot_path = os.path.join(RESULTS_ROOT, 'cv_metrics.png')
    plt.savefig(plot_path, bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Plot saved to: {plot_path}')
else:
    print('No metrics to plot.')

## Step 7: Heatmap Generation

Generate per-WSI spatial prediction heatmaps.

**When `USE_HF_EMBEDDINGS = True` (no raw WSI files):**
A synthetic gray canvas is used instead of the tissue thumbnail. Patches are colored and spatially positioned by their prediction coordinates. The result shows the spatial pattern of predicted subtypes without the tissue background.

> **Note on coordinate accuracy:** If the HuggingFace `.npy` files contain real `(x, y)` pixel coordinates in their dict (check `--inspect` output), the patch positions will be spatially accurate. If only synthetic coordinates were available (`x=patch_index, y=0`), patches will appear as a 1D strip — not a meaningful spatial map. Real coordinates are required for accurate heatmaps.

Color scheme: CC → Red, EC → Green, HGSC → Yellow, LGSC → Blue, MC → Purple

In [ ]:
# Select which fold's model to use for heatmaps
# Change this to use a specific fold, or loop over all folds
HEATMAP_FOLD = 0

s2_ckpt_hm = os.path.join(S2_OUT, f'fold{HEATMAP_FOLD}', 'best_model.pth')
inst_csv_dir = os.path.join(S2_OUT, f'fold{HEATMAP_FOLD}')

if not os.path.isfile(s2_ckpt_hm):
    raise FileNotFoundError(f'Stage 2 checkpoint not found: {s2_ckpt_hm}')
print(f'Using checkpoint: {s2_ckpt_hm}')
print(f'Instance predictions dir: {inst_csv_dir}')

In [ ]:
heatmap_cmd = [
    sys.executable, 'scripts/07_generate_heatmaps.py',
    '--predictions_dir', inst_csv_dir,
    '--output_dir',      HEATMAP_OUT,
    '--labels_csv',      LABELS_CSV,
    '--gaussian_sigma',  '5',
]

if not USE_HF_EMBEDDINGS and os.path.isdir(WSI_DIR):
    # Raw WSI files available — use tissue thumbnail as background
    heatmap_cmd += ['--wsi_dir', WSI_DIR]
    print('Heatmap mode: tissue thumbnail background (WSI files available)')
else:
    # No WSI files — use synthetic canvas (patch layout only)
    print('Heatmap mode: synthetic canvas (no WSI files — USE_HF_EMBEDDINGS=True)')
    print('Patches will be positioned by their prediction coordinates.')

run(heatmap_cmd)
print('\n✓ Heatmaps generated.')

In [ ]:
from IPython.display import Image, display
import glob as _glob

heatmap_files = sorted(_glob.glob(os.path.join(HEATMAP_OUT, '*_heatmap.png')))
print(f'Generated {len(heatmap_files)} heatmaps.')

# Display first 4 heatmaps inline
N_SHOW = min(4, len(heatmap_files))
for hf in heatmap_files[:N_SHOW]:
    print(os.path.basename(hf))
    display(Image(hf, width=700))

## Summary

In [ ]:
import subprocess as _sp

print('=== Output File Tree ===')
_sp.run(['find', RESULTS_ROOT, '-name', '*.pth', '-o',
                                '-name', 'metrics.json', '-o',
                                '-name', '*_heatmap.png', '-o',
                                '-name', 'aggregate_metrics.csv'],
        check=False)

print('\n=== Final Metrics ===')
agg_path = os.path.join(RESULTS_ROOT, 'aggregate_metrics.csv')
if os.path.isfile(agg_path):
    df_agg = pd.read_csv(agg_path, index_col=0)
    print(df_agg.to_string(float_format='{:.4f}'.format))
    print('\nMean ± Std:')
    for col in df_agg.columns:
        mu  = df_agg[col].mean()
        std = df_agg[col].std()
        print(f'  {col:20s}: {mu*100:.2f} ± {std*100:.2f} %')
else:
    print('aggregate_metrics.csv not found — run Step 6 first.')